<a href="https://colab.research.google.com/github/syntheticbeek/bravo-baseball-model/blob/main/COLABPREDICTIVEMODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Build your training dataset
data = {
    'cast_member': [
        'Denise Richards', 'Raquel Leviss', 'Olivia Flowers',
        'Taylor Ann Green', 'Kyle Richards',
        'Tom Sandoval', 'Austen Kroll', 'Whitney Sudler-Smith'
    ],
    'show': [
        'RHOBH', 'VPR', 'Southern Charm',
        'Southern Charm', 'RHOBH',
        'VPR', 'Southern Charm', 'Southern Charm'
    ],
    'trigger_timing': [
        'mid', 'late', 'early',
        'early', 'early',
        'late', 'early', 'mid'
    ],
    'confirmed': [0, 0, 1, 0, 0, 1, 0, 0],
    'cast_reaction': [2, 2, 1, 2, 1, 2, 1, 1],
    'betrayal_index': [8, 10, 6.5, 8, 5, 10, 7, 6],
    'narrative_equity': [7, 7, 7, 8, 10, 10, 8, 5],
    'victim_narrative_index': [8, 8, 7, 9, 8, 1, 1, 3],
    'prior_incident': [0, 0, 1, 0, 0, 1, 1, 0],
    'gender': [0, 0, 0, 0, 0, 1, 1, 1],  # 0=woman, 1=man
    'outcome': [1, 1, 1, 1, 0, 0, 0, 0]  # 1=exited, 0=stayed
}

df = pd.DataFrame(data)

# Encode trigger timing
timing_map = {'early': 0, 'mid': 1, 'late': 2, 'after_filming': 3}
df['trigger_timing_encoded'] = df['trigger_timing'].map(timing_map)

print(df[['cast_member', 'gender', 'betrayal_index', 'victim_narrative_index', 'outcome']])

# Features for training
features = ['trigger_timing_encoded', 'confirmed', 'cast_reaction',
            'betrayal_index', 'narrative_equity', 'victim_narrative_index',
            'prior_incident', 'gender']

X_train = df[features]
y_train = df['outcome']

# Train the model
model = LogisticRegression()
model.fit(X_train, y_train)

# Print model weights
print("\n--- Variable Weights ---")
weights = pd.DataFrame({
    'Variable': features,
    'Weight': model.coef_[0]
}).sort_values('Weight', ascending=False)
print(weights.to_string(index=False))

# Amanda and West prediction data
test_data = {
    'cast_member': ['Amanda', 'West'],
    'trigger_timing_encoded': [3, 3],      # after filming
    'confirmed': [0, 0],                    # both denied
    'cast_reaction': [2, 2],               # both hostile
    'betrayal_index': [10, 6],
    'narrative_equity': [10, 8],
    'victim_narrative_index': [10, 3],
    'prior_incident': [0, 0],
    'gender': [0, 1]                        # Amanda=woman, West=man
}

test_df = pd.DataFrame(test_data)
predictions = model.predict_proba(test_df[features])

print("\n--- Predictions ---")
for i, name in enumerate(['Amanda', 'West']):
    print(f"{name} — Exit probability: {predictions[i][1]:.1%}")

            cast_member  gender  betrayal_index  victim_narrative_index  \
0       Denise Richards       0             8.0                       8   
1         Raquel Leviss       0            10.0                       8   
2        Olivia Flowers       0             6.5                       7   
3      Taylor Ann Green       0             8.0                       9   
4         Kyle Richards       0             5.0                       8   
5          Tom Sandoval       1            10.0                       1   
6          Austen Kroll       1             7.0                       1   
7  Whitney Sudler-Smith       1             6.0                       3   

   outcome  
0        1  
1        1  
2        1  
3        1  
4        0  
5        0  
6        0  
7        0  

--- Variable Weights ---
              Variable    Weight
victim_narrative_index  0.785028
        betrayal_index  0.610436
             confirmed  0.218178
        prior_incident  0.204283
         cast_re

In [7]:
# Remove victim narrative index from features and retrain
features_no_victim = ['trigger_timing_encoded', 'confirmed', 'cast_reaction',
                      'betrayal_index', 'narrative_equity',
                      'prior_incident', 'gender']

model_no_victim = LogisticRegression()
model_no_victim.fit(df[features_no_victim], df['outcome'])

# Rerun Amanda and West without victim narrative
test_no_victim = pd.DataFrame([{
    'trigger_timing_encoded': 3,
    'confirmed': 0,
    'cast_reaction': 2,
    'betrayal_index': 10,
    'narrative_equity': 10,
    'prior_incident': 0,
    'gender': 0
},
{
    'trigger_timing_encoded': 3,
    'confirmed': 0,
    'cast_reaction': 2,
    'betrayal_index': 6,
    'narrative_equity': 8,
    'prior_incident': 0,
    'gender': 1
}])

preds_no_victim = model_no_victim.predict_proba(test_no_victim[features_no_victim])

print("--- Without Victim Narrative Variable ---")
for i, name in enumerate(['Amanda', 'West']):
    print(f"{name} — Exit probability: {preds_no_victim[i][1]:.1%}")

print(f"\nGap without victim narrative: {abs(preds_no_victim[0][1] - preds_no_victim[1][1]):.1%}")

# Also run without betrayal index for comparison
features_no_betrayal = ['trigger_timing_encoded', 'confirmed', 'cast_reaction',
                        'narrative_equity', 'victim_narrative_index',
                        'prior_incident', 'gender']

model_no_betrayal = LogisticRegression()
model_no_betrayal.fit(df[features_no_betrayal], df['outcome'])

test_no_betrayal = pd.DataFrame([{
    'trigger_timing_encoded': 3,
    'confirmed': 0,
    'cast_reaction': 2,
    'narrative_equity': 10,
    'victim_narrative_index': 10,
    'prior_incident': 0,
    'gender': 0
},
{
    'trigger_timing_encoded': 3,
    'confirmed': 0,
    'cast_reaction': 2,
    'narrative_equity': 8,
    'victim_narrative_index': 3,
    'prior_incident': 0,
    'gender': 1
}])

preds_no_betrayal = model_no_betrayal.predict_proba(test_no_betrayal[features_no_betrayal])

print("\n--- Without Betrayal Index Variable ---")
for i, name in enumerate(['Amanda', 'West']):
    print(f"{name} — Exit probability: {preds_no_betrayal[i][1]:.1%}")

print(f"\nGap without betrayal index: {abs(preds_no_betrayal[0][1] - preds_no_betrayal[1][1]):.1%}")

# Summary comparison
print("\n--- Summary: Gap Under Each Condition ---")
print(f"Full model (all variables):     94.1%")
print(f"Without gender:                 94.2%")
print(f"Without victim narrative:       {abs(preds_no_victim[0][1] - preds_no_victim[1][1]):.1%}")
print(f"Without betrayal index:         {abs(preds_no_betrayal[0][1] - preds_no_betrayal[1][1]):.1%}")

--- Without Victim Narrative Variable ---
Amanda — Exit probability: 43.8%
West — Exit probability: 8.6%

Gap without victim narrative: 35.2%

--- Without Betrayal Index Variable ---
Amanda — Exit probability: 87.9%
West — Exit probability: 5.2%

Gap without betrayal index: 82.7%

--- Summary: Gap Under Each Condition ---
Full model (all variables):     94.1%
Without gender:                 94.2%
Without victim narrative:       35.2%
Without betrayal index:         82.7%
